# Studi Kasus Webinar ke-5: Human-Computer Interaction & Data Visualization
## Dashboard Analisis Kepuasan Pengunjung Perpustakaan

Notebook ini mengimplementasikan dashboard untuk mengevaluasi pengalaman pengunjung perpustakaan menggunakan **Python, Pandas, NumPy, Plotly, dan ipywidgets**.

### Tujuan
- Menganalisis data simulasi kepuasan pengunjung perpustakaan.
- Menampilkan ringkasan statistik dan visualisasi sesuai kebutuhan analisis.
- Menerapkan prinsip HCI: *overview first*, *recognition over recall*, konsistensi, umpan balik, dan filter interaktif.
- Menyediakan pencarian, ekspor data, serta checklist pengujian usability.

> **Catatan:** Dataset bersifat sintetis untuk pembelajaran, bukan hasil survei perpustakaan nyata.

## 1. Instalasi pustaka
Jalankan sel berikut jika pustaka yang diperlukan belum tersedia.

In [ ]:
%pip install -q pandas numpy plotly ipywidgets

## 2. Import pustaka

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
print("Pustaka berhasil diimpor.")

## 3. Membuat dataset demonstrasi
Data mencakup ID kunjungan, kelompok pengunjung, jenis layanan, frekuensi kunjungan, waktu tunggu, rating kepuasan, dan durasi kunjungan.

In [ ]:
rng = np.random.default_rng(42)
jumlah = 300
kelompok = ["Mahasiswa", "Dosen", "Tenaga Kependidikan", "Umum"]
layanan = ["Peminjaman Buku", "Ruang Baca", "Referensi", "Akses Komputer"]
df = pd.DataFrame({
    "ID Kunjungan": [f"KJG{i:04d}" for i in range(1, jumlah + 1)],
    "Kelompok Pengunjung": rng.choice(kelompok, jumlah, p=[0.58, 0.16, 0.10, 0.16]),
    "Jenis Layanan": rng.choice(layanan, jumlah),
    "Frekuensi Kunjungan/Bulan": rng.integers(1, 17, jumlah),
    "Waktu Tunggu (menit)": np.clip(rng.gamma(2.2, 3.0, jumlah), 0.5, 25).round(1),
    "Rating Kepuasan": rng.choice([1, 2, 3, 4, 5], jumlah, p=[0.04, 0.08, 0.20, 0.38, 0.30]),
    "Durasi Kunjungan (menit)": np.clip(rng.normal(65, 25, jumlah), 10, 180).round(1)
})
df["Kategori Kepuasan"] = pd.cut(
    df["Rating Kepuasan"], bins=[0, 2, 3, 4, 6],
    labels=["Rendah", "Cukup", "Baik", "Sangat Baik"], right=True
)
display(df.head(10))
print(f"Jumlah baris: {len(df)}")
print(f"Jumlah kolom: {len(df.columns)}")

## 4. Memahami dan memeriksa data
Pemeriksaan awal membantu memastikan data siap dianalisis.

In [ ]:
print("Informasi dataset:")
df.info()
print("\\nJumlah nilai kosong per kolom:")
display(df.isna().sum().to_frame("Nilai kosong"))
print("\\nStatistik deskriptif:")
display(df[["Waktu Tunggu (menit)", "Rating Kepuasan", "Durasi Kunjungan (menit)"]].describe().round(2))

## 5. Ringkasan statistik (*overview first*)
Tampilkan informasi utama sebelum pengguna mengeksplorasi grafik dan detail.

In [ ]:
ringkasan = pd.DataFrame({
    "Metrik": ["Jumlah kunjungan", "Rata-rata rating", "Rata-rata waktu tunggu", "Rata-rata durasi kunjungan", "Persentase rating 4–5"],
    "Nilai": [
        len(df), round(df["Rating Kepuasan"].mean(), 2),
        f'{df["Waktu Tunggu (menit)"].mean():.1f} menit',
        f'{df["Durasi Kunjungan (menit)"].mean():.1f} menit',
        f'{df["Rating Kepuasan"].ge(4).mean()*100:.1f}%'
    ]
})
display(ringkasan)

## 6. Visualisasi 1 — Rata-rata kepuasan per layanan
**Bar chart** membandingkan rata-rata rating setiap jenis layanan.

In [ ]:
rating_layanan = (df.groupby("Jenis Layanan", as_index=False)["Rating Kepuasan"]
    .mean().sort_values("Rating Kepuasan", ascending=False))
fig_bar = px.bar(rating_layanan, x="Jenis Layanan", y="Rating Kepuasan",
    text="Rating Kepuasan", title="Rata-rata Kepuasan per Jenis Layanan",
    labels={"Rating Kepuasan": "Rata-rata rating"})
fig_bar.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig_bar.update_layout(yaxis_range=[0, 5.5], showlegend=False)
fig_bar.show()
display(rating_layanan.round(2))

## 7. Visualisasi 2 — Distribusi kategori kepuasan
**Donut chart** menunjukkan proporsi kategori rating.

In [ ]:
urutan = ["Rendah", "Cukup", "Baik", "Sangat Baik"]
distribusi = (df["Kategori Kepuasan"].value_counts().reindex(urutan, fill_value=0)
    .rename_axis("Kategori Kepuasan").reset_index(name="Jumlah"))
fig_donut = px.pie(distribusi, names="Kategori Kepuasan", values="Jumlah",
    hole=0.45, title="Distribusi Kategori Kepuasan")
fig_donut.update_traces(textposition="inside", textinfo="percent+label")
fig_donut.show()
display(distribusi)

## 8. Visualisasi 3 — Hubungan waktu tunggu dan kepuasan
**Scatter plot** membantu mengeksplorasi hubungan dua variabel. Pola yang terlihat tidak otomatis membuktikan sebab-akibat.

In [ ]:
fig_scatter = px.scatter(df, x="Waktu Tunggu (menit)", y="Rating Kepuasan",
    color="Jenis Layanan", hover_data=["ID Kunjungan", "Kelompok Pengunjung", "Durasi Kunjungan (menit)"],
    title="Hubungan Waktu Tunggu dan Rating Kepuasan",
    labels={"Rating Kepuasan": "Rating kepuasan (1–5)"})
fig_scatter.update_layout(yaxis_range=[0.5, 5.5])
fig_scatter.show()

## 9. Visualisasi 4 — Rata-rata kepuasan berdasarkan frekuensi kunjungan
Line chart memperlihatkan variasi rating menurut kelompok frekuensi kunjungan.

In [ ]:
df["Kelompok Frekuensi"] = pd.cut(df["Frekuensi Kunjungan/Bulan"],
    bins=[0, 3, 7, 11, 20], labels=["1–3 kali", "4–7 kali", "8–11 kali", "12+ kali"])
rating_frekuensi = (df.groupby("Kelompok Frekuensi", observed=False, as_index=False)["Rating Kepuasan"].mean())
fig_line = px.line(rating_frekuensi, x="Kelompok Frekuensi", y="Rating Kepuasan",
    markers=True, title="Rata-rata Kepuasan berdasarkan Frekuensi Kunjungan",
    labels={"Rating Kepuasan": "Rata-rata rating"})
fig_line.update_layout(yaxis_range=[0, 5.5])
fig_line.show()
display(rating_frekuensi.round(2))

## 10. Filter interaktif (*zoom and filter*)
Pilih kelompok pengunjung, jenis layanan, dan rentang rating. Jika widget tidak tampil pada lingkungan tertentu, visualisasi sebelumnya tetap dapat digunakan.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

kelompok_widget = widgets.SelectMultiple(options=sorted(df["Kelompok Pengunjung"].unique()),
    value=tuple(sorted(df["Kelompok Pengunjung"].unique())), description="Pengunjung:",
    layout=widgets.Layout(width="350px", height="100px"))
layanan_widget = widgets.SelectMultiple(options=sorted(df["Jenis Layanan"].unique()),
    value=tuple(sorted(df["Jenis Layanan"].unique())), description="Layanan:",
    layout=widgets.Layout(width="350px", height="100px"))
rating_widget = widgets.IntRangeSlider(value=(1, 5), min=1, max=5, step=1,
    description="Rentang rating:", layout=widgets.Layout(width="500px"))
output_filter = widgets.Output()

def update_dashboard(change=None):
    with output_filter:
        clear_output(wait=True)
        filtered = df[
            df["Kelompok Pengunjung"].isin(kelompok_widget.value)
            & df["Jenis Layanan"].isin(layanan_widget.value)
            & df["Rating Kepuasan"].between(rating_widget.value[0], rating_widget.value[1])
        ].copy()
        display(Markdown("### Ringkasan hasil filter"))
        if filtered.empty:
            print("Tidak ada data yang sesuai. Ubah pilihan filter.")
            return
        display(pd.DataFrame({"Metrik": ["Jumlah kunjungan", "Rata-rata rating", "Rata-rata waktu tunggu"],
            "Nilai": [len(filtered), round(filtered["Rating Kepuasan"].mean(), 2),
                      f'{filtered["Waktu Tunggu (menit)"].mean():.1f} menit']}))
        by_service = filtered.groupby("Jenis Layanan", as_index=False)["Rating Kepuasan"].mean().sort_values("Rating Kepuasan", ascending=False)
        fig = px.bar(by_service, x="Jenis Layanan", y="Rating Kepuasan", text="Rating Kepuasan",
            title="Rata-rata rating setelah filter")
        fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
        fig.update_layout(yaxis_range=[0, 5.5], showlegend=False)
        fig.show()
        display(Markdown("### Detail kunjungan"))
        display(filtered.sort_values("Rating Kepuasan", ascending=False).reset_index(drop=True))

for widget in [kelompok_widget, layanan_widget, rating_widget]:
    widget.observe(update_dashboard, names="value")
display(widgets.VBox([kelompok_widget, layanan_widget, rating_widget, output_filter]))
update_dashboard()

## 11. Pencarian detail kunjungan
Pencarian membantu pengguna menemukan ID kunjungan atau kelompok pengunjung tanpa menelusuri seluruh tabel.

In [ ]:
kata_kunci = input("Masukkan ID kunjungan atau kelompok pengunjung (kosongkan untuk semua): ").strip()
if kata_kunci:
    hasil_cari = df[
        df["ID Kunjungan"].str.contains(kata_kunci, case=False, na=False)
        | df["Kelompok Pengunjung"].str.contains(kata_kunci, case=False, na=False)
    ]
else:
    hasil_cari = df.copy()
print(f"Jumlah hasil: {len(hasil_cari)}")
display(hasil_cari.sort_values("Rating Kepuasan", ascending=False).head(30))

## 12. Ekspor data hasil analisis
Ekspor dataset dan hasil filter ke CSV untuk analisis lanjutan.

In [ ]:
df.to_csv("data_kepuasan_perpustakaan_demo.csv", index=False, encoding="utf-8-sig")
filtered_export = df[
    df["Kelompok Pengunjung"].isin(kelompok_widget.value)
    & df["Jenis Layanan"].isin(layanan_widget.value)
    & df["Rating Kepuasan"].between(rating_widget.value[0], rating_widget.value[1])
]
filtered_export.to_csv("data_kepuasan_perpustakaan_hasil_filter.csv", index=False, encoding="utf-8-sig")
print("File berhasil dibuat:")
print("- data_kepuasan_perpustakaan_demo.csv")
print("- data_kepuasan_perpustakaan_hasil_filter.csv")

## 13. Checklist pengujian usability

Lakukan pengujian bersama pengguna dan isi hasilnya berdasarkan percobaan nyata.

| No. | Skenario pengujian | Kriteria keberhasilan |
|---|---|---|
| 1 | Menemukan jumlah kunjungan dan rata-rata rating | Informasi utama mudah ditemukan |
| 2 | Memilih kelompok pengunjung | Ringkasan dan grafik menyesuaikan filter |
| 3 | Mengubah rentang rating | Data di luar rentang tidak ditampilkan |
| 4 | Mencari ID kunjungan | Hasil sesuai kata kunci |
| 5 | Mengekspor CSV | File dapat dibuka dan isinya sesuai |

### Prinsip HCI yang diterapkan
- **Overview first:** ringkasan statistik ditampilkan sebelum detail.
- **Zoom and filter:** pengguna mempersempit data melalui filter.
- **Details on demand:** detail ditampilkan setelah fokus dipilih.
- **Recognition over recall:** judul dan label menjelaskan informasi grafik.
- **Feedback:** hasil diperbarui setelah filter atau pencarian.
- **Konsistensi:** format angka, judul, dan label seragam.

## 14. Kesimpulan

Notebook ini menerapkan prinsip HCI dan visualisasi data untuk mengeksplorasi kepuasan pengunjung perpustakaan. Bar chart digunakan untuk membandingkan layanan, donut chart menunjukkan proporsi kategori kepuasan, scatter plot mengeksplorasi hubungan waktu tunggu dan rating, sedangkan line chart memperlihatkan variasi menurut frekuensi kunjungan. Filter, pencarian, dan ekspor mendukung eksplorasi data.

Dataset bersifat sintetis dan tidak mewakili survei nyata. Untuk keputusan operasional, gunakan data yang valid, uji usability bersama pengguna, serta jaga privasi responden.